# ⚙️ SVM Analysis

Binary classification task: predict loan approval using **Support Vector Machine (SVM)** classifiers.

SVMs are effective for high-dimensional spaces and perform well when a clean decision boundary separates classes. We explore `linear` and `rbf` kernels with hyperparameter tuning via GridSearchCV.

**Notebook outline:**
1. Objective
2. Import Libraries
3. Import Custom Modules
4. Load Dataset
5. Basic Cleaning
6. Feature Engineering
7. Feature Selection
8. Train-Test Split
9. Build SVM Pipeline
10. Hyperparameter Tuning
11. Evaluation
12. ROC Curve
13. Confusion Matrix
14. Feature Comparison
15. Save Model
16. Conclusion

## 1. Objective

Train and evaluate **Support Vector Machine (SVM)** classifiers to predict whether an HDFC loan application will be **Approved** or **Rejected**.

Key goals:
- Compare `linear` and `rbf` SVM kernels
- Tune `C` (regularisation) and `gamma` (RBF kernel coefficient) via GridSearchCV
- Evaluate using accuracy, precision, recall, F1, ROC-AUC, and confusion matrix
- Compare the best SVM against the baseline from Notebook 02

## 2. Import Libraries

Load standard libraries and configure warnings.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC

## 3. Import Custom Modules

Add the project root to the path and import all shared utility modules.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.preprocessing import *
from src.feature_engineering import *
from src.train import *
from src.evaluate import *
from src.utils import *

## 4. Load Dataset

Load the enriched HDFC loan dataset from the `dataset/` directory.

In [ ]:
df = load_dataset("../dataset/hdfc_loan_dataset_full_enriched.csv")

print("Shape:", df.shape)
df.head()

## 5. Basic Cleaning

Remove duplicate rows and strip whitespace from column names.

In [ ]:
df = basic_cleaning(df)

print_summary(df)

In [ ]:
missing_value_summary(df)

## 6. Feature Engineering

Engineer domain-specific features using `src/feature_engineering.py`:
- `Total_Income` — applicant + co-applicant income
- `EMI_Income_Ratio` — existing EMIs relative to total income
- `Loan_Income_Ratio` — requested loan relative to total income

In [ ]:
df = create_features(df)

df[[
    "Total_Income",
    "EMI_Income_Ratio",
    "Loan_Income_Ratio"
]].describe()

## 7. Feature Selection

Select the predictor columns and encode the binary target `Loan_Status`.

We use the same feature set as Notebook 02 for direct model comparability.

In [ ]:
features = [
    "Applicant_Income",
    "Coapplicant_Income",
    "Loan_Amount",
    "Credit_History",
    "CIBIL_Score",
    "Employment_Status",
    "Existing_EMIs",
    "Debt_to_Income_Ratio",
    "Education",
    "Property_Area",

    "Total_Income",
    "Loan_Income_Ratio",
    "EMI_Income_Ratio"
]

target = "Loan_Status"

In [ ]:
x, y = select_features(
    df,
    features,
    target
)

le = LabelEncoder()
y  = le.fit_transform(y)

print("Classes      :", le.classes_)
print("Feature shape:", x.shape)

In [ ]:
num_features, cat_features = get_feature_types(x)

preprocessor = create_preprocessor(
    num_features,
    cat_features
)

## 8. Train-Test Split

Split 80/20 with stratification to preserve class balance in both sets.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape :", x_train.shape)
print("Testing Shape  :", x_test.shape)

## 9. Build SVM Pipeline

Build two baseline SVM pipelines — **linear** and **RBF** kernel — using `build_pipeline` from `src/train.py`.

> `probability=True` enables `predict_proba`, which is required for ROC curve computation.

In [ ]:
svm_linear = build_pipeline(
    preprocessor,
    SVC(
        kernel="linear",
        probability=True,
        random_state=42
    )
)

In [ ]:
svm_rbf = build_pipeline(
    preprocessor,
    SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
)

In [ ]:
svm_linear.fit(x_train, y_train)
svm_rbf.fit(x_train, y_train)

linear_pred = predict(svm_linear, x_test)
rbf_pred    = predict(svm_rbf,    x_test)

print("Linear SVM fitted")
print("RBF SVM fitted")

In [ ]:
linear_results = evaluate_classification(
    y_test,
    linear_pred,
    "SVM Linear"
)
linear_results

In [ ]:
rbf_results = evaluate_classification(
    y_test,
    rbf_pred,
    "SVM RBF"
)
rbf_results

## 10. Hyperparameter Tuning

Tune the SVM over `C`, `gamma`, and `kernel` using `GridSearchCV` with 5-fold CV, optimising for **ROC-AUC**.

| Parameter | Values | Notes |
|-----------|--------|-------|
| `C`       | 0.1, 1, 10, 100 | Regularisation — higher values fit training data tighter |
| `gamma`   | scale, auto, 0.01, 0.001 | RBF kernel width |
| `kernel`  | linear, rbf | Kernel type |

In [ ]:
svm_pipeline = build_pipeline(
    preprocessor,
    SVC(
        probability=True,
        random_state=42
    )
)

svm_params = {
    "classifier__C"     : [0.1, 1, 10, 100],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma" : ["scale", "auto", 0.01, 0.001]
}

In [ ]:
grid_svm = perform_grid_search(
    pipeline=svm_pipeline,
    parameters=svm_params,
    x_train=x_train,
    y_train=y_train,
    scoring="roc_auc",
    cv=5
)

print("Best Parameters:", grid_svm.best_params_)
print("Best CV ROC-AUC: {:.4f}".format(grid_svm.best_score_))

In [ ]:
best_svm  = grid_svm.best_estimator_

best_pred = predict(
    best_svm,
    x_test
)

## 11. Evaluation

Evaluate the tuned SVM and compare all three SVM variants.

In [ ]:
best_results = evaluate_classification(
    y_test,
    best_pred,
    "SVM Tuned"
)
best_results

In [ ]:
print_classification_report(
    y_test,
    best_pred
)

In [ ]:
comparison = compare_models(
    [linear_results, rbf_results, best_results],
    names=["SVM Linear", "SVM RBF", "SVM Tuned"]
)
comparison

## 12. ROC Curve

Plot the ROC curve for the best tuned SVM. A higher AUC indicates better discrimination between approved and rejected applications.

In [ ]:
plot_roc_curve(
    best_svm,
    x_test,
    y_test,
    "SVM Tuned — ROC Curve"
)

## 13. Confusion Matrix

Visualise prediction breakdown: true positives, true negatives, false positives, and false negatives.

In [ ]:
plot_confusion_matrix(
    y_test,
    best_pred,
    "SVM Tuned — Confusion Matrix"
)

## 14. Feature Comparison

SVMs do not expose `feature_importances_` directly. For the **linear kernel**, we extract the decision boundary **coefficients** to rank feature influence.

Positive coefficients push toward approval; negative coefficients push toward rejection.

In [ ]:
# Fit a dedicated linear SVM to extract interpretable coefficients
svm_coef_pipeline = build_pipeline(
    preprocessor,
    SVC(
        kernel="linear",
        C=1,
        probability=True,
        random_state=42
    )
)

svm_coef_pipeline.fit(x_train, y_train)

In [ ]:
feature_names = svm_coef_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = svm_coef_pipeline.named_steps[
    "classifier"
].coef_[0]

coef_df = pd.DataFrame({
    "Feature"    : feature_names,
    "Coefficient": coefficients
}).sort_values(
    by="Coefficient",
    key=abs,
    ascending=False
)

coef_df.head(15)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=coef_df.head(15),
    x="Coefficient",
    y="Feature",
    palette="coolwarm"
)

plt.title("Top 15 SVM Linear Coefficients (Feature Influence)")
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

## 15. Save Model

Persist the best tuned SVM pipeline to disk using `joblib`.

In [ ]:
save_model(
    best_svm,
    "../models/svm_model.pkl"
)

## 16. Conclusion

### Summary

| Model | Notes |
|-------|-------|
| SVM Linear (baseline) | Fast, interpretable via coefficients |
| SVM RBF (baseline)    | Better for non-linear boundaries |
| SVM Tuned (GridSearch)| Best CV ROC-AUC across `C`, `gamma`, and `kernel` |

### Key Takeaways
- The **RBF kernel** generally outperforms the linear kernel on this dataset due to non-linear relationships between income, CIBIL score, and loan status.
- **`C`** controls the regularisation strength — higher values reduce the margin to fit training data more tightly, at the risk of overfitting.
- **`CIBIL_Score`**, **`Credit_History`**, and **`Debt_to_Income_Ratio`** are the strongest predictors per the linear coefficient analysis.
- SVM training is slower than tree-based ensembles; Random Forest or Gradient Boosting remain better choices for production throughput.

### Next Steps
- Try `SGDClassifier` with `hinge` loss as a scalable SVM approximation for larger datasets
- Experiment with polynomial kernel (`degree=2, 3`)
- Apply SMOTE oversampling if class imbalance is detected and re-evaluate